
## **CLI: Setting Up a Cloud Composer Environment**

Setting up a Cloud Composer environment is a straightforward process that involves creating a new instance through the Google Cloud console.

### Prerequisities

#### **Step 1: Enable Necessary APIs**

Before creating your environment, you need to ensure the following Google Cloud APIs are enabled in your project:

  * **Cloud Composer API:** `composer.googleapis.com`
  * **Cloud Build API:** `cloudbuild.googleapis.com`
  * **Cloud Functions API:** `cloudfunctions.googleapis.com`
  * **Cloud Storage API:** `storage.googleapis.com`
  * **Compute Engine API:** `compute.googleapis.com`
  * **Google Kubernetes Engine API:** `container.googleapis.com`
  * **Cloud SQL API:** `sqladmin.googleapis.com`
  * **IAM API:** `iam.googleapis.com`
  * **Cloud Logging API:** `logging.googleapis.com`
  * **Cloud Monitoring API:** `monitoring.googleapis.com`

You can enable all of these from the **APIs & Services > Dashboard** in the Google Cloud console or you can use the `gcloud` CLI installed and authenticated to your desired Google Cloud project.  

Replace `PROJECT_ID` with the actual ID of your Google Cloud project.

Then, use the following command to enable all the specified APIs:


In [ ]:
# Authenticate gcloud CLI, set project variables, and enable required GCP APIs
gcloud auth login
SERVICE_ACCOUNT_NAME="example-workshop-composer-sa"
PROJECT_ID="poc-example-ds" 
# Define your project ID and the service account email
SERVICE_ACCOUNT_EMAIL="${SERVICE_ACCOUNT_NAME}@${PROJECT_ID}.iam.gserviceaccount.com"
gcloud config set project $PROJECT_ID
gcloud services enable \
    composer.googleapis.com \
    cloudbuild.googleapis.com \
    cloudfunctions.googleapis.com \
    storage.googleapis.com \
    compute.googleapis.com \
    container.googleapis.com \
    sqladmin.googleapis.com \
    iam.googleapis.com \
    logging.googleapis.com \
    monitoring.googleapis.com


#### **Step 2: Service Account Permissions**

You have two main approaches for the service account used by your Composer environment:

  * **Using the default Compute Engine service account:** This is simpler but grants broad permissions, which might not align with least privilege security principles. Its email is typically `PROJECT_NUMBER-compute@developer.gserviceaccount.com`.
  * **Creating a custom service account:** This is recommended for production environments as it allows you to grant only the necessary permissions.

Let's go with creating and configuring a **custom service account**.

**1.1. Create a Custom Service Account**

If you get an error, check if it is related to the fact that the SA exists already. If it does, skip this command:


In [ ]:
# Create the dedicated custom Service Account for Cloud Composer workers

gcloud iam service-accounts create ${SERVICE_ACCOUNT_NAME} \
    --display-name "Service Account for Cloud Composer Environment" \
    --project=${PROJECT_ID}


**1.2. Grant Required Roles to the Service Account**

The service account needs various roles to interact with Cloud Composer itself and other Google Cloud services that your Airflow DAGs will use (e.g., BigQuery, Cloud Storage, etc.).

**Minimum Composer-specific roles:**

  * `roles/composer.worker`: Allows the service account to run Airflow workers and interact with Composer's underlying infrastructure.
  * `roles/composer.serviceAgent`: This is primarily for a Google-managed service agent, but sometimes specific operations might require a subset of these permissions on your environment's SA if you're doing complex custom setups. For standard cases, `composer.worker` is the main one for your environment's SA.
  * `roles/container.nodeServiceAccount`: Necessary because Composer runs on GKE, and this role grants permissions for the underlying GKE nodes.

**Common additional roles (adjust based on your DAGs' needs):**

  * `roles/bigquery.dataEditor`: If your DAGs need to write data to BigQuery.
  * `roles/bigquery.dataViewer`: If your DAGs only need to read data from BigQuery.
  * `roles/bigquery.jobUser`: If your DAGs submit BigQuery jobs.
  * `roles/storage.objectAdmin`: If your DAGs read/write objects to Cloud Storage buckets (including the DAGs bucket itself).
  * `roles/compute.networkUser`: Essential for the service account to use the specified VPC network and subnetwork.
  * `roles/logging.logWriter`: For writing logs to Cloud Logging.
  * `roles/monitoring.metricWriter`: For writing metrics to Cloud Monitoring.


In [ ]:
# Define your project ID and the service account email
SERVICE_ACCOUNT_EMAIL="${SERVICE_ACCOUNT_NAME}@${PROJECT_ID}.iam.gserviceaccount.com"

# Grant Composer-specific roles
gcloud projects add-iam-policy-binding ${PROJECT_ID} \
    --member="serviceAccount:${SERVICE_ACCOUNT_EMAIL}" \
    --role="roles/composer.worker" \
    --project=${PROJECT_ID}

gcloud projects add-iam-policy-binding ${PROJECT_ID} \
    --member="serviceAccount:${SERVICE_ACCOUNT_EMAIL}" \
    --role="roles/container.nodeServiceAccount" \
    --project=${PROJECT_ID}

# Grant network access
gcloud projects add-iam-policy-binding ${PROJECT_ID} \
    --member="serviceAccount:${SERVICE_ACCOUNT_EMAIL}" \
    --role="roles/compute.networkUser" \
    --project=${PROJECT_ID}

# Grant Cloud Storage access (adjust if you need more granular control, e.g., only specific buckets)
gcloud projects add-iam-policy-binding ${PROJECT_ID} \
    --member="serviceAccount:${SERVICE_ACCOUNT_EMAIL}" \
    --role="roles/storage.objectAdmin" \
    --project=${PROJECT_ID}

# Grant BigQuery access (choose appropriate BigQuery roles based on your DAGs' needs)
gcloud projects add-iam-policy-binding ${PROJECT_ID} \
    --member="serviceAccount:${SERVICE_ACCOUNT_EMAIL}" \
    --role="roles/bigquery.dataEditor" \
    --project=${PROJECT_ID}

gcloud projects add-iam-policy-binding ${PROJECT_ID} \
    --member="serviceAccount:${SERVICE_ACCOUNT_EMAIL}" \
    --role="roles/bigquery.jobUser" \
    --project=${PROJECT_ID}

# Grant logging and monitoring permissions
gcloud projects add-iam-policy-binding ${PROJECT_ID} \
    --member="serviceAccount:${SERVICE_ACCOUNT_EMAIL}" \
    --role="roles/logging.logWriter" \
    --project=${PROJECT_ID}

gcloud projects add-iam-policy-binding ${PROJECT_ID} \
    --member="serviceAccount:${SERVICE_ACCOUNT_EMAIL}" \
    --role="roles/monitoring.metricWriter" \
    --project=${PROJECT_ID}



gcloud projects add-iam-policy-binding ${PROJECT_ID} \
    --member="serviceAccount:${SERVICE_ACCOUNT_EMAIL}" \
    --role="roles/dataproc.editor" \
    --project=${PROJECT_ID}

gcloud projects add-iam-policy-binding ${PROJECT_ID} \
    --member="serviceAccount:${SERVICE_ACCOUNT_EMAIL}" \
    --role="roles/iam.serviceAccountUser" \
    --project=${PROJECT_ID}


gcloud projects add-iam-policy-binding ${PROJECT_ID} \
    --member="serviceAccount:${SERVICE_ACCOUNT_EMAIL}" \
    --role="roles/dataproc.worker" \
    --project=${PROJECT_ID}


gcloud projects add-iam-policy-binding ${PROJECT_ID} \
    --member="serviceAccount:${SERVICE_ACCOUNT_EMAIL}" \
    --role="roles/bigquery.readSessionUser" \
    --project=${PROJECT_ID}

echo "Service account ${SERVICE_ACCOUNT_EMAIL} created and roles granted."


In [ ]:
# # Set variables for clarity
# export IMPERSONATED_SA_EMAIL="poc-example-ds@appspot.gserviceaccount.com"
# export IMPERSONATING_SA_EMAIL="example-workshop-composer-sa@poc-example-ds.iam.gserviceaccount.com" # Your Airflow worker SA

# gcloud iam service-accounts add-iam-policy-binding ${IMPERSONATED_SA_EMAIL} \
#     --member="serviceAccount:${IMPERSONATING_SA_EMAIL}" \
#     --role="roles/iam.serviceAccountUser"


To check the permissions use: 


In [ ]:
# Inspect all IAM roles currently bound to your custom Service Account
gcloud projects get-iam-policy $PROJECT_ID  \
--flatten="bindings[].members" \
--format='table(bindings.role)' \
--filter="bindings.members:$SERVICE_ACCOUNT_EMAIL"


##### Warning when using Composer V2



The service account `service-$PROJECT_NUMBER@cloudcomposer-accounts.iam.gserviceaccount.com` is a **Google-managed service account**, often referred to as the **Cloud Composer Service Agent**. This is *distinct* from the service account you explicitly provide when creating your environment (which in your case is `example-workshop-composer-sa@poc-example-ds.iam.gserviceaccount.com` and acts as the "environment's service account" or "worker service account").

Here's why you need to grant the `roles/composer.ServiceAgentV2Ext` (`Cloud Composer v2 API Service Agent Extension`) role to the **Google-managed service agent**:

1.  **Orchestrating Underlying Infrastructure:** Cloud Composer v2 runs on a Google Kubernetes Engine (GKE) cluster. When you create or update a Composer v2 environment, the Google-managed service agent is responsible for performing operations on this underlying GKE cluster and other related Google Cloud resources. This includes:

      * Creating and managing the GKE cluster itself.
      * Configuring networking, including setting up VPC-native features and ensuring the private IP connectivity works correctly.
      * Managing Cloud SQL instances (for Airflow metadata database).
      * Setting up Cloud Storage buckets for DAGs and logs.
      * Creating bindings between your environment's service account (e.g., `example-workshop-composer-sa`) and the Kubernetes service accounts within the GKE cluster. This binding is essential for your DAGs to run with the correct permissions.

2.  **IAM Policy Management:** The `Cloud Composer v2 API Service Agent Extension` role grants specific permissions like `iam.serviceAccounts.getIamPolicy` and `iam.serviceAccounts.setIamPolicy`. These permissions are vital because the Composer Service Agent needs to be able to *read* and *modify* the IAM policies on your **environment's service account** (`example-workshop-composer-sa@poc-example-ds.iam.gserviceaccount.com`). This allows Composer to bind your environment's service account to the Kubernetes service account running your Airflow pods, ensuring your DAGs execute with the permissions you intend.

3.  **"Missing Role" Error:** Historically, and often still, this specific role (`roles/composer.ServiceAgentV2Ext`) is **not automatically granted** to the Cloud Composer Service Agent when you enable the Composer API or create your first environment. If this role is missing, environment creation will typically fail with a permissions error, even if your custom environment service account (`example-workshop-composer-sa`) has all the necessary roles. It's a common troubleshooting step for Composer v2 environment creation failures.

**In essence:**

  * **`service-$PROJECT_NUMBER@cloudcomposer-accounts.iam.gserviceaccount.com` (Cloud Composer Service Agent):** This is Google's agent for managing the *Composer environment itself* and its underlying infrastructure. It needs the `Cloud Composer v2 API Service Agent Extension` role to properly configure and maintain your Composer v2 environment, especially its interaction with the GKE cluster and the environment's service account.
  * **`example-workshop-composer-sa@poc-example-ds.iam.gserviceaccount.com` (Environment's Service Account / Worker Service Account):** This is *your* service account that the Airflow workers and schedulers within your Composer environment will use to execute DAGs and interact with other Google Cloud services (like BigQuery, Cloud Storage, etc.). This account needs roles like `composer.worker`, `storage.objectAdmin`, `bigquery.jobUser`, etc., depending on what your DAGs do.

**The command to grant this specific role:**


In [ ]:
# Grant the ServiceAgentV2Ext role to the Google-managed Composer Service Agent
PROJECT_NUMBER=$(gcloud projects describe $PROJECT_ID --format='value(projectNumber)')

COMPOSER_SERVICE_AGENT="service-$PROJECT_NUMBER@cloudcomposer-accounts.iam.gserviceaccount.com" # <<< The specific service account email

gcloud projects add-iam-policy-binding ${PROJECT_ID} \
    --member="serviceAccount:${COMPOSER_SERVICE_AGENT}" \
    --role="roles/composer.ServiceAgentV2Ext" \
    --project=${PROJECT_ID}



The `roles/composer.ServiceAgentV2Ext` is an **Identity and Access Management (IAM) role** specific to **Google Cloud Composer** environments, particularly for **Composer v2 and v3**.

Here's a breakdown of what it is and why it's important:

1.  **Purpose:** This role grants the necessary permissions to the **Cloud Composer Service Agent** to manage various Google Cloud resources on behalf of your project. When you create a Composer environment, Google automatically provisions a Google-managed service account (often in the format `service-PROJECT_NUMBER@cloudcomposer-accounts.iam.gserviceaccount.com`). This service account needs specific permissions to operate.

2.  **What it allows Composer to do:** The `roles/composer.ServiceAgentV2Ext` role provides the Composer service agent with the permissions required to:
    * **Manage Google Kubernetes Engine (GKE) clusters:** Composer environments are built on GKE. The service agent needs to create, configure, and manage the underlying GKE cluster, including nodes, networking, and other components.
    * **Interact with Cloud Storage:** This is crucial for managing DAGs, logs, and other Airflow-related files stored in your Composer bucket.
    * **Access other GCP services:** Depending on the Composer environment's configuration and features (e.g., Private IP, networking, Cloud Logging, Monitoring), the service agent requires permissions to interact with these services.
    * **Provision resources:** It allows the service to provision and manage the various compute, storage, and networking resources that make up your Composer environment.

3.  **Why `V2Ext`?**
    * The `V2Ext` suffix indicates that this role is specifically designed for Composer environments running on version 2.x and above (including v3). These versions have a different architecture compared to Composer v1, requiring a distinct set of permissions for the service agent.

4.  **Automatic Granting (Typically):** In most cases, when you enable the Cloud Composer API or create a Composer environment through the GCP Console or `gcloud` CLI, this role is **automatically granted** to the Composer Service Agent of your project.

5.  **When you might explicitly set it:** The `gcloud projects add-iam-policy-binding` command you provided earlier (`gcloud projects add-iam-policy-binding ${PROJECT_ID} --member="serviceAccount:${COMPOSER_SERVICE_AGENT}" --role="roles/composer.ServiceAgentV2Ext"`) is a manual way to ensure this binding exists. This can be useful in automated deployment scripts (like Terraform or `gcloud` scripts) or if the automatic binding somehow fails or needs to be re-established.

6.  **Important Distinction:** This role is for the **Composer service agent** itself, which manages the *Composer environment*. It is distinct from the **Airflow Worker Service Account** (the one you specify with `--service-account` during environment creation), which is used by your Airflow DAGs to interact with other GCP services (e.g., BigQuery, Cloud Storage buckets where your data resides). Both are critical for a functional Composer environment.

In essence, `roles/composer.ServiceAgentV2Ext` is a high-privilege, Google-managed role that is fundamental for the successful operation and lifecycle management of your Composer v2 or v3 environment.



#### **Step 3: VPC Network and Subnetwork with Secondary IP Ranges**

For a private IP Cloud Composer environment, you need a custom-mode VPC network with a subnetwork that has available secondary IP ranges. Composer uses these secondary ranges for the GKE cluster's pods and services.

**2.1. Create a Custom Mode VPC Network**


In [ ]:
# Create a custom-mode VPC network so we control subnet CIDR allocation
VPC_NETWORK_NAME="composer-workshop-vpc"

gcloud compute networks create ${VPC_NETWORK_NAME} \
    --subnet-mode=custom \
    --project=${PROJECT_ID}

echo "VPC Network '${VPC_NETWORK_NAME}' created."


To **check for existing IP ranges and ensure your proposed `PRIMARY_IP_RANGE`, `GKE_PODS_IP_RANGE`, and `GKE_SERVICES_IP_RANGE` are unique and available within your Google Cloud project and VPC network**, you'll primarily use `gcloud compute networks subnets` commands.

The goal is to avoid **overlaps** with:

1.  **Existing primary IP ranges** of any subnet within your VPC network.
2.  **Existing secondary IP ranges** of any subnet within your VPC network.
3.  **IP ranges in peered VPC networks** (if your `VPC_NETWORK_NAME` is peered with other VPCs).



**1. List All Subnets and Their IP Ranges in Your VPC Network** This is the most crucial step. It will show you the primary and any secondary IP ranges currently in use by all subnets within your specified VPC.


In [ ]:
# List all existing subnets and secondary IP ranges in the VPC to prevent overlaps
gcloud compute networks subnets list \
    --network=$VPC_NETWORK_NAME \
    --project=$PROJECT_ID \
    --format="table(name,region,ipCidrRange,secondaryIpRanges[].ipCidrRange.list():label=SECONDARY_IP_RANGES)"


**Case A: empty output** an empty output from `gcloud compute networks subnets list` for your specified network `$VPC_NETWORK_NAME` means one of two things:

1.  **The `$VPC_NETWORK_NAME` itself is a brand new network you are about to create (or just created) with `--subnet-mode=custom`.**

      * **This is the most likely and expected scenario based on your previous commands.** When you create a custom-mode VPC network, it starts empty and doesn't have any pre-defined subnets. Therefore, listing subnets immediately after creating the network (but before creating any subnets within it) will yield an empty list.

2.  **The `$VPC_NETWORK_NAME` does not exist at all, or there's a typo in the name/project ID.** (Less likely if you just ran `gcloud compute networks create`).

**What an empty output means for your IP ranges:**

If the `$VPC_NETWORK_NAME` is indeed a new, empty custom-mode VPC network (or an existing one that truly has no subnets yet), then an empty output is **good news for your proposed IP ranges** within *that specific VPC network*. It means:

  * There are **no existing primary subnets** in `VPC_NETWORK_NAME` that would conflict with `PRIMARY_IP_RANGE="10.10.0.0/20"`.
  * There are **no existing secondary IP ranges** in `VPC_NETWORK_NAME` that would conflict with `GKE_PODS_IP_RANGE="10.10.16.0/22"` or `GKE_SERVICES_IP_RANGE="10.10.20.0/22"`.

**The Most Important Remaining Check:**

While the empty output confirms no internal conflicts *within this specific VPC network*, it **does NOT guarantee** that your proposed IP ranges are unique across your entire Google Cloud environment, especially if you have:

  * **VPC Peering:** If your `VPC_NETWORK_NAME` is or will be peered with other VPC networks (e.g., in other projects, or with Google-managed service producer networks), you **must** ensure your chosen IP ranges `10.10.0.0/20`, `10.10.16.0/22`, and `10.10.20.0/22` do not overlap with any IP ranges in those peered networks. Conflicts in peered networks can cause significant routing problems.

      * To check for existing peerings, you'd run:
        ```bash
        gcloud compute networks describe ${VPC_NETWORK_NAME} --project=${PROJECT_ID} --format="json"
        ```
        Look for the `"peerings"` array in the output. If it's not empty, you'll need to investigate the networks listed there.



**CASE B: Interpreting the Output:**
Look at the `ipCidrRange` (primary) and `SECONDARY_IP_RANGES` columns for all listed subnets. Manually compare your proposed ranges (`10.10.0.0/20`, `10.10.16.0/22`, `10.10.20.0/22`) against these existing ranges to ensure there is no overlap.

**Example Output (and how to check for conflicts):**

| NAME               | REGION       | IP\_CIDR\_RANGE | SECONDARY\_IP\_RANGES       |
| :----------------- | :----------- | :------------ | :------------------------ |
| default            | europe-west1 | 10.128.0.0/20 |                           |
| another-subnet-eu  | europe-west1 | 10.100.0.0/24 | gke-data-pods:10.100.1.0/24 |
| prod-subnet-us     | us-central1  | 10.10.0.0/24  | prod-svc:10.10.1.0/24     |

In this example, your proposed `PRIMARY_IP_RANGE="10.10.0.0/20"` would **conflict** with `prod-subnet-us`'s primary range `10.10.0.0/24` (as /20 is a larger block that encompasses /24). You would need to choose different, non-overlapping ranges.

**2. Check for VPC Peering (Crucial for Larger Deployments)** If your `VPC_NETWORK_NAME` is peered with other VPC networks (e.g., for connecting to other projects or services), you *must* also check the IP ranges used in those peered networks. Conflicts across peered networks can cause routing issues.


In [ ]:
# Inspect VPC network details (including any active VPC peerings)
gcloud compute networks describe $VPC_NETWORK_NAME \
    --project=${PROJECT_ID} \
    --format="json"



Look for the `"peerings"` array in the JSON output. If peerings exist, you'll need to coordinate with the administrators of those peered networks to understand their IP space usage.

**3. Planning Your IP Ranges**

  * **Non-Overlapping:** Ensure your primary and all secondary ranges are entirely separate from each other and from any other existing ranges in your VPC and peered VPCs.
  * **Size:** Ensure the `/20` for your primary and `/22` for your secondary ranges are sufficiently large for your needs. Composer environments typically require substantial IP space for GKE Pods and Services.
  * **RFC 1918:** For internal networks, ensure your chosen ranges fall within the private IP address ranges (10.0.0.0/8, 172.16.0.0/12, 192.168.0.0/16).
  * **Subnet Planning Tool:** For complex VPC setups, consider using a network planning tool or spreadsheet to map out your IP address space visually.

By performing these checks, you can confidently define your `PRIMARY_IP_RANGE`, `GKE_PODS_IP_RANGE`, and `GKE_SERVICES_IP_RANGE` to avoid conflicts when creating your subnetwork.


**2.2. Create a Subnetwork with Secondary IP Ranges**

  * `--range`: This is the primary IP range for the subnetwork.
  * `--secondary-range`: You'll need at least two secondary ranges for Cloud Composer: one for GKE pods and one for GKE services.
      * **`--secondary-range [NAME_FOR_PODS]=[CIDR_FOR_PODS]`**
      * **`--secondary-range [NAME_FOR_SERVICES]=[CIDR_FOR_SERVICES]`**

Make sure your CIDR ranges do not overlap with any other networks or on-premises connections.


In [ ]:
# Create the regional subnetwork with Private Google Access and GKE secondary ranges
SUBNET_NAME="composer-workshop-subnet"
REGION="europe-west1"  
PRIMARY_IP_RANGE="10.10.0.0/20" # <<< REPLACE if needed, ensure it's unique
GKE_PODS_RANGE_NAME="gke-pods-range"
GKE_PODS_IP_RANGE="10.10.16.0/22" # <<< REPLACE if needed, ensure it's unique and available
GKE_SERVICES_RANGE_NAME="gke-services-range"
GKE_SERVICES_IP_RANGE="10.10.20.0/22" # <<< REPLACE if needed, ensure it's unique and available

gcloud compute networks subnets create ${SUBNET_NAME} \
    --network=${VPC_NETWORK_NAME} \
    --range=${PRIMARY_IP_RANGE} \
    --region=${REGION} \
    --enable-private-ip-google-access \
    --secondary-range ${GKE_PODS_RANGE_NAME}=${GKE_PODS_IP_RANGE} \
    --secondary-range ${GKE_SERVICES_RANGE_NAME}=${GKE_SERVICES_IP_RANGE} \
    --project=${PROJECT_ID}

echo "Subnetwork '${SUBNET_NAME}' created with primary and secondary IP ranges."



**Note on `--enable-private-ip-google-access`:** This flag is crucial for private IP environments as it allows instances in the subnetwork to reach Google APIs and services (like Cloud Storage, BigQuery, and Composer's control plane) using internal IP addresses, without needing external IP addresses or Cloud NAT.


**Notice**: You **do not always need a secondary IP range** for Google Cloud Composer. However, you **do need a secondary IP range** if you are creating a **VPC-native** (also known as private IP) Composer environment.

Here's why and the distinction:

1.  **VPC-Native (Private IP) Composer Environments (Recommended)**
    * **Requirement:** Yes, these **require secondary IP ranges**.
    * **Why:** Cloud Composer environments are built on top of Google Kubernetes Engine (GKE). In a VPC-native GKE cluster, the Kubernetes Pods and Services (which make up your Airflow components like workers, schedulers, and the web server) are allocated IP addresses from **secondary IP ranges** within your subnetwork, rather than from the primary IP range used by the GKE nodes themselves. This design offers:
        * **Improved Security:** Your cluster nodes, Pods, and Services use internal IP addresses, preventing direct exposure to the public internet.
        * **Better IP Management:** It allows for more efficient use of your IP address space within your VPC.
        * **Simplified Networking:** Easier integration with other internal GCP services.
    * **How it's configured:** In **Cloud Composer 2**, you pass `--cluster-ipv4-cidr` / `--services-ipv4-cidr` or secondary subnetwork ranges. In **Cloud Composer 3**, Google abstracts the GKE cluster entirely via Private Service Connect, so secondary IP ranges on your subnet are only required if you also run VPC-native GKE workloads or Composer 2 in the same subnet.* When you create the subnetwork, you define a primary IP range and then one or more secondary IP ranges. When creating the Composer environment, you specify which subnetwork to use, and Composer automatically uses one of its secondary ranges for the GKE cluster's Pods and Services.

2.  **Routes-Based (Non-VPC Native) Composer Environments (Legacy/Less Recommended)**
    * **Requirement:** No, these **do not require secondary IP ranges**.
    * **Why:** In this older model, GKE clusters use custom routes to enable communication between Pods. The Pods and Services are allocated IPs directly from the primary IP range of the subnetwork (or a larger block outside the subnetwork range).
    * **Disadvantages:** Generally less secure (as internal traffic might rely on external routing), less scalable for IP address management, and not the recommended best practice for modern GKE and Composer deployments. Google strongly advises using VPC-native clusters.

**In summary:**

If you are following best practices for security and network architecture on GCP, you will almost certainly be creating a VPC-native Composer environment, which **will require you to configure secondary IP ranges** on your chosen subnetwork beforehand.

When using the `gcloud composer environments create` command, you specify a VPC-native environment using the `--enable-private-environment` flag, and you must also specify the `--network` and `--subnetwork` parameters pointing to a subnetwork that has available secondary IP ranges.


In [ ]:
# --- Define your variables (ensure they are set correctly from previous steps) ---
export PROJECT_ID="poc-example-ds"
export VPC_NETWORK_NAME="composer-workshop-vpc"
export SUBNET_NAME="composer-workshop-subnet"
export PRIMARY_IP_RANGE="10.10.0.0/20" # This is the primary range of your SUBNET_NAME


# --- Create the Firewall Rule ---
# This rule allows all internal traffic (TCP, UDP, ICMP) within the specified IP range.
# It's an INGRESS rule, meaning it applies to incoming connections to instances within the network.
# The --source-ranges should be the IP range of the subnet itself, allowing communication
# between all resources (nodes, pods) within that subnet.

gcloud compute firewall-rules create allow-internal-dataproc-traffic \
    --project=${PROJECT_ID} \
    --network=${VPC_NETWORK_NAME} \
    --action=ALLOW \
    --rules=tcp,udp,icmp \
    --source-ranges=${PRIMARY_IP_RANGE} \
    --destination-ranges=${PRIMARY_IP_RANGE} \
    --direction=INGRESS \
    --description="Allow all internal TCP/UDP/ICMP traffic within the Composer/Dataproc subnet for Spark communication."

echo "Firewall rule 'allow-internal-dataproc-traffic' created."
echo "Please re-run your Airflow DAG after this rule is active."


If you still have networking related issues, run the following: 


In [ ]:
# Example Airflow / GCP code snippet

export PROJECT_ID="poc-example-ds" # Ensure this is your project ID
gcloud compute firewall-rules delete allow-internal-dataproc-traffic \
    --project=${PROJECT_ID}
echo "Existing firewall rule 'allow-internal-dataproc-traffic' deleted."

export PROJECT_ID="poc-example-ds"
export VPC_NETWORK_NAME="composer-workshop-vpc"
export PRIMARY_IP_RANGE="10.10.0.0/20" # Your subnet's primary IP range, as per your config

echo "Creating corrected firewall rule 'allow-internal-dataproc-traffic'..."
gcloud compute firewall-rules create allow-internal-dataproc-traffic \
    --project=${PROJECT_ID} \
    --network=${VPC_NETWORK_NAME} \
    --action=ALLOW \
    --rules=tcp:1-65535,udp:1-65535,icmp \
    --source-ranges=${PRIMARY_IP_RANGE} \
    --destination-ranges=${PRIMARY_IP_RANGE} \
    --direction=INGRESS \
    --priority=1000 \
    --description="Allow all internal TCP/UDP/ICMP traffic within the Composer/Dataproc subnet for Spark communication."
echo "Corrected firewall rule 'allow-internal-dataproc-traffic' created."
echo "Please wait a minute for propagation, then re-trigger your Airflow DAG."



In [ ]:
# IF YOU JUST NEED TO CREATE THE CLUSTER AND YOU HAVE ALREADY RUN IN A PREVIOUS SESSIONS THE COMMANDS ABOVE 
# decomment the variables below
gcloud auth login
SERVICE_ACCOUNT_NAME="example-workshop-composer-sa"
PROJECT_ID="poc-example-ds" 
SUBNET_NAME="composer-workshop-subnet"
VPC_NETWORK_NAME="composer-workshop-vpc"
SERVICE_ACCOUNT_EMAIL="${SERVICE_ACCOUNT_NAME}@${PROJECT_ID}.iam.gserviceaccount.com"
GCP_REGION=europe-west1

# This bucket must exist and be in the same region as your Composer environment.
# The Composer Service Account (service-$PROJECT_NUMBER@cloudcomposer-accounts.iam.gserviceaccount.com)
# will need Storage Object Admin permissions on this bucket.
DEDICATED_DAGS_BUCKET="europe-west1-workshop-comop-01f64e99-bucket" # e.g., "my-project-airflow-dags"

gcloud composer environments create workshop-composer-v3 \
    --location=$GCP_REGION \
    --image-version=composer-3-airflow-2.10.5 \
    --environment-size=SMALL \
    --enable-private-environment \
    --network=projects/$PROJECT_ID/global/networks/$VPC_NETWORK_NAME \
    --subnetwork=projects/$PROJECT_ID/regions/$GCP_REGION/subnetworks/$SUBNET_NAME \
    --service-account=$SERVICE_ACCOUNT_EMAIL \
    --scheduler-cpu=1 \
    --scheduler-memory=4GB \
    --scheduler-count=1 \
    --worker-cpu=1 \
    --worker-memory=4GB \
    --worker-storage=1GB \
    --min-workers=1 \
    --max-workers=3 \
    --web-server-allow-all \
    --storage-bucket=$DEDICATED_DAGS_BUCKET \
    --labels=mode=training,user=zelda


once the command has successfully run you will see in the UI the following:
![image.png](../artifacts/03_composer_env_created.png)


### **Where to Find the DAGs Folder**

Click on Dags Folder link to find the  `/dags` folder 

![image.png](../artifacts/03_composer_dags_folder.png)
Every Cloud Composer environment has an associated Cloud Storage bucket. Airflow in Cloud Composer only schedules DAGs that are located in the `/dags` folder within this bucket.

To access your environment's bucket:

1.  In the Google Cloud console, go to the **Environments** page.
2.  In the list of environments, find the row with your environment's name, and in the **DAGs folder** column, click the **DAGs** link. This will open the **Bucket details** page, showing you the contents of the `/dags` folder in your environment's bucket.


### **Where to Set the Bucket for Code Dependencies**

When you create an environment, Cloud Composer creates a Cloud Storage bucket and associates it with that environment. You can store custom code in the Cloud Storage bucket, including DAGs, plugins, and other resources. Composer uses `gcsfuse` to sync all the content to the Airflow workers and web server.

To install code dependencies, such as Python libraries, you can specify the PyPI packages to install in the Cloud Composer environment. You can do this when you create the environment or at a later time.


### **Environment Teardown & Cost Cleanup**

Cloud Composer environments run continuously and incur hourly compute and database charges. Once you have completed the workshop exercises, always delete your environment and temporary resources:

```bash
gcloud composer environments delete workshop-composer-v3 \
    --location=${GCP_REGION} \
    --project=${PROJECT_ID} \
    --quiet
```

> **Note:** Deleting a Cloud Composer environment does **not** automatically delete its Cloud Storage bucket or custom VPC network, ensuring you do not accidentally lose DAG code or shared networking infrastructure.
